In [1]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from transformers import AutoImageProcessor, AutoModel
import torch.nn as nn
import os
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from tqdm.notebook import tqdm
from icecream import ic
from transformers import get_cosine_schedule_with_warmup
from PIL import Image
import random
import numpy as np
import timm
import cv2

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)



/home/system/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/system/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias

In [2]:

from vit_tiny import vit_tiny_classifier
model = vit_tiny_classifier(n_classes=1000)
model = model.to('cuda')

In [3]:
class DatasetImagenet(Dataset):
    def __init__(self, root_dir, split='ILSVRC2012_img_train', transform=None):
        self.root_dir = root_dir
        self.split = split
        self.transform = transform

        self.samples = []
        self.transform = transforms.Compose([

                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
            ])

        if split == 'ILSVRC2012_img_train':
            train_dir = os.path.join(root_dir, split)
            self.classes = sorted(os.listdir(train_dir))
            self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}

            for cls_name in tqdm(self.classes):
                cls_path = os.path.join(train_dir, cls_name)
                for img_name in os.listdir(cls_path):
                    image_path = os.path.join(cls_path, img_name)
                    self.samples.append((image_path, self.class_to_idx[cls_name]))

        else:
            val_dir = os.path.join(root_dir, split)
            gt_file = os.path.join(root_dir, 'ILSVRC2012_validation_ground_truth.txt')

            with open(gt_file) as f:
                gt_labels = [int(x.strip()) - 1 for x in f.readlines()]

            img_files = sorted(os.listdir(val_dir))

            for img_name, label in zip(img_files, gt_labels):
                image_path = os.path.join(val_dir, img_name)
                self.samples.append((image_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = Image.open(path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label
train_dataset = DatasetImagenet(root_dir='/media/system/ZERBUIS_EXT_STOR/dynamic_slam/imagenet', split='ILSVRC2012_img_train')
train_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True) # total of 12 million sampels

test_dataset = DatasetImagenet(root_dir='/media/system/ZERBUIS_EXT_STOR/dynamic_slam/imagenet', split='ILSVRC2012_img_val')
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False) # 50k samples

  0%|          | 0/1000 [00:00<?, ?it/s]

In [4]:
LR = 5e-4
EPOCHS = 300
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.05

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999)
)

device = 'cuda'
model = model.to(device)

num_training_steps = EPOCHS * len(train_dataloader)
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

criterion = nn.CrossEntropyLoss()
output_dir = 'vit_tiny_checkpoints/'
os.makedirs(output_dir, exist_ok=True)


In [5]:

for epoch in tqdm(range(EPOCHS)):

    # ---------------- TRAIN ----------------
    model.train()
    total_loss = 0
    correct, total = 0, 0

    for images, labels in tqdm(train_dataloader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)           # (B, num_classes)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        # accuracy
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    avg_loss = total_loss / len(train_dataloader)

    # ---------------- VALID ----------------
    model.eval()
    val_correct, val_total = 0, 0

    if epoch < 10 or epoch % 10 == 0:
        with torch.no_grad():
            for images, labels in test_dataloader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                _, predicted = torch.max(outputs, 1)

                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = 100 * val_correct / val_total
        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Loss: {avg_loss:.4f} "
            f"Train Acc: {train_acc:.2f}% "
            f"Val Acc: {val_acc:.2f}%"
        )

    else:
        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Loss: {avg_loss:.4f} "
            f"Train Acc: {train_acc:.2f}% "
        )
    # Save checkpoint
    if epoch%5==0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            # 'val_acc': val_acc,
        }, f"{output_dir}/checkpoint_epoch_{epoch}.pt")

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/5005 [00:00<?, ?it/s]

KeyboardInterrupt: 